# PKL Diffusion Denoising - Google Colab Training

This notebook provides a complete setup for training diffusion models on Google Colab with A100 GPU, including live monitoring and visualization.

## Features
- 🚀 Optimized for A100 GPU (batch size 32, mixed precision)
- 🛡️ Browser disconnection crash prevention
- 💾 Automatic checkpointing and resuming
- 📊 Real-time monitoring with TensorBoard
- 🖼️ Live image visualization per epoch
- 🔄 Memory optimization and batch size adaptation
- 📈 Training on 11,475 real microscopy pairs (9,000 train / 1,125 val / 1,350 test)
- 🎯 Complete dataset included in GitHub repository (~92M)
- ⏰ Max epochs: 1000 (typical diffusion training: 800-1000 epochs)
- 🎯 Expected training time: 8-12 hours (full training to convergence)


## 1. Environment Setup & Installation


In [ ]:
# Optimized GPU check and environment setup
import os
import subprocess

# Quick GPU check
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_name = result.stdout.strip()

print(f"🚀 GPU: {gpu_name}")
print(f"📁 Working directory: {os.getcwd()}")

# Set optimal environment variables immediately
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'  # Optimize memory allocation

if 'A100' in gpu_name:
    print("✅ A100 GPU detected - optimal configuration enabled!")
    print("📊 Batch size: 32 | ⚡ Mixed precision: 16-bit | 🔄 Max epochs: 1000")
    print("⏰ Expected training time: 8-12 hours")
else:
    print("⚠️ A100 not detected - using standard configuration")


Thu Sep 11 02:56:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Optimized setup: Mount Drive and install packages in parallel
from google.colab import drive
import subprocess
import threading

def mount_drive():
    """Mount Google Drive."""
    print("📁 Mounting Google Drive...")
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")

def install_packages():
    """Install required packages efficiently."""
    print("📦 Installing packages...")
    
    # Install PyTorch with CUDA support
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', 
                   '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
    
    # Install other packages in one command
    subprocess.run(['pip', 'install', '-q', 
                   'hydra-core', 'omegaconf', 'wandb', 'diffusers', 'accelerate',
                   'pytorch-lightning', 'tensorboard', 'pillow', 'numpy', 
                   'tqdm', 'zarr', 'numcodecs', 'matplotlib', 'seaborn'], check=True)
    
    print("✅ Packages installed successfully.")

# Run both operations
mount_drive()
install_packages()

# Set project path
colab_project_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
print(f"📂 Project path: {colab_project_path}")

Attempting to mount Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


## 2. Project Setup & Configuration

In [ ]:
# 🚀 ENHANCED CPU OPTIMIZATIONS FOR COLAB A100
import multiprocessing
import os

# Detect available CPU cores for optimal thread allocation
cpu_count = multiprocessing.cpu_count()
print(f"🖥️  Detected {cpu_count} CPU cores")

# Optimize CPU threads for different libraries
os.environ['OMP_NUM_THREADS'] = str(min(8, cpu_count))  # OpenMP threads
os.environ['MKL_NUM_THREADS'] = str(min(4, cpu_count))  # Intel MKL threads  
os.environ['NUMEXPR_NUM_THREADS'] = str(min(4, cpu_count))  # NumExpr threads
os.environ['OPENBLAS_NUM_THREADS'] = str(min(4, cpu_count))  # OpenBLAS threads

# PyTorch CPU optimizations
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Avoid tokenizer warnings
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512,expandable_segments:True,garbage_collection_threshold:0.8'

# Optimal num_workers calculation for Colab
optimal_num_workers = min(8, max(2, cpu_count // 2))  # Conservative for Colab
print(f"📊 Optimal num_workers: {optimal_num_workers}")

# Additional CPU optimizations
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # Async CUDA operations
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '1'  # Enable cuDNN v8 API

print("✅ Enhanced CPU optimizations applied!")
print(f"🔧 OMP_NUM_THREADS: {os.environ['OMP_NUM_THREADS']}")
print(f"🔧 MKL_NUM_THREADS: {os.environ['MKL_NUM_THREADS']}")
print(f"🔧 Optimal num_workers: {optimal_num_workers}")


In [ ]:
# 🔧 DYNAMIC CONFIG OPTIMIZATION
# Update training config with optimal num_workers
from omegaconf import OmegaConf

# Load the training config
training_cfg_path = f'{colab_project_path}/configs/training/ddpm_colab.yaml'
training_cfg = OmegaConf.load(training_cfg_path)

# Update num_workers with optimal value
training_cfg.num_workers = optimal_num_workers
print(f"📊 Updated num_workers to: {training_cfg.num_workers}")

# Save updated config
OmegaConf.save(training_cfg, training_cfg_path)
print("✅ Training config updated with optimal CPU settings!")

# Display optimized configuration
print("\n🚀 OPTIMIZED TRAINING CONFIGURATION:")
print(f"📊 Batch size: {training_cfg.batch_size}")
print(f"🔧 Num workers: {training_cfg.num_workers}")
print(f"⚡ Mixed precision: {training_cfg.precision}")
print(f"🔄 Max epochs: {training_cfg.max_epochs}")
print(f"💾 Persistent workers: {training_cfg.persistent_workers}")
print(f"📈 Prefetch factor: {training_cfg.prefetch_factor}")


In [ ]:
# 🚀 ADDITIONAL PERFORMANCE OPTIMIZATIONS
import torch
import gc

# PyTorch optimizations
torch.backends.cudnn.benchmark = True  # Optimize for consistent input sizes
torch.backends.cudnn.deterministic = False  # Allow non-deterministic for speed
torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 for A100
torch.backends.cudnn.allow_tf32 = True  # Allow TF32 for cuDNN

# Memory optimizations
torch.cuda.empty_cache()  # Clear GPU cache
gc.collect()  # Clear Python garbage

# Enable memory efficient attention if available
try:
    torch.backends.cuda.enable_flash_sdp(True)
    print("✅ Flash Attention enabled")
except:
    print("⚠️  Flash Attention not available")

# Display GPU memory info
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_memory_free = torch.cuda.memory_reserved(0) / 1024**3
    print(f"🎮 GPU Memory: {gpu_memory:.1f}GB total")
    print(f"🎮 GPU Memory Free: {gpu_memory_free:.1f}GB")

print("✅ Additional performance optimizations applied!")
print("🔧 cuDNN benchmark: Enabled")
print("🔧 TF32: Enabled for A100")
print("🔧 Memory cache: Cleared")


In [ ]:
# Optimized project setup and configuration loading
import os
from omegaconf import OmegaConf

# Set environment variables for optimal performance
os.environ['PROJECT_ROOT'] = colab_project_path
os.environ['OMP_NUM_THREADS'] = '4'  # Optimize CPU threads
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Avoid tokenizer warnings

print("🔧 Setting up project and loading configuration...")

# Install project package efficiently
print("📦 Installing project package...")
os.chdir(colab_project_path)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
os.chdir('/content')

# Load and verify configuration
try:
    cfg = OmegaConf.load(f'{colab_project_path}/configs/config_colab.yaml')
    training_cfg = OmegaConf.load(f'{colab_project_path}/configs/training/ddpm_colab.yaml')
    
    print("✅ Configuration loaded successfully!")
    print(f"📊 Batch size: {training_cfg.batch_size} | Precision: {training_cfg.precision}")
    print(f"🔄 Max epochs: {training_cfg.max_epochs} | Conditioning: {training_cfg.use_conditioning}")
    print(f"💾 Artifacts saved to: {colab_project_path}")
    
except Exception as e:
    print(f"❌ Configuration error: {e}")
    print(f"Make sure configs exist in: {colab_project_path}/configs/")

Project code path set to: /content/drive/MyDrive/PKL-DiffusionDenoising


In [ ]:
# Optimized TensorBoard setup with live monitoring
import subprocess
import threading
import time
from pathlib import Path

def start_tensorboard():
    """Start TensorBoard efficiently for live monitoring."""
    logs_dir = f'{colab_project_path}/logs'
    os.makedirs(logs_dir, exist_ok=True)
    
    print(f"📊 Starting TensorBoard (logs: {logs_dir})")
    
    # Start TensorBoard with optimized settings
    cmd = ['tensorboard', '--logdir', logs_dir, '--port', '6006', '--host', '0.0.0.0', 
           '--reload_interval', '30', '--max_reload_threads', '4']
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    time.sleep(2)  # Reduced wait time
    
    if process.poll() is None:
        print("✅ TensorBoard started successfully!")
        print("📊 Live monitoring available at:")
        print("   https://colab.research.google.com/github/tensorflow/tensorboard/blob/master/tensorboard/notebooks/tensorboard_in_notebooks.ipynb")
        return process
    else:
        print("❌ Failed to start TensorBoard")
        return None

# Start TensorBoard
tb_process = start_tensorboard()


In [ ]:
## 3. Live Monitoring & Visualization


In [ ]:
# Live monitoring functions (for use AFTER training or between training sessions)
import glob
import matplotlib.pyplot as plt
from PIL import Image
import torch

def monitor_training_progress():
    """Check training progress - use this between training sessions."""
    checkpoint_dir = f'{colab_project_path}/checkpoints'
    samples_dir = f'{colab_project_path}/outputs/samples'
    
    print("📊 Training Progress Monitor")
    print("=" * 50)
    
    # Check checkpoints efficiently
    checkpoints = glob.glob(f"{checkpoint_dir}/epoch_*.pt")
    if checkpoints:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f"💾 Latest: {os.path.basename(latest_ckpt)}")
        
        # Quick checkpoint info without full loading
        try:
            ckpt_data = torch.load(latest_ckpt, map_location='cpu', weights_only=True)
            print(f"📈 Epoch: {ckpt_data.get('epoch', '?')} | Loss: {ckpt_data.get('last_val_loss', '?'):.4f}")
        except:
            print("📈 Checkpoint info unavailable")
    else:
        print("⏳ No checkpoints found")
    
    # Check for latest samples
    if os.path.exists(samples_dir):
        sample_files = glob.glob(f"{samples_dir}/*/epoch_*.png")
        if sample_files:
            latest_sample = max(sample_files, key=os.path.getctime)
            print(f"🎨 Latest sample: {os.path.basename(latest_sample)}")
    
    print("=" * 50)

def display_latest_validation():
    """Display latest validation images - use this between training sessions."""
    samples_dir = f'{colab_project_path}/outputs/samples'
    
    sample_dirs = glob.glob(f"{samples_dir}/*")
    if not sample_dirs:
        print("❌ No sample directories found")
        return
    
    latest_dir = max(sample_dirs, key=os.path.getctime)
    val_files = glob.glob(f"{latest_dir}/validation_epoch_*.png")
    
    if not val_files:
        print("❌ No validation images found")
        return
    
    # Show only the latest 2 images for speed
    latest_vals = sorted(val_files, key=os.path.getctime)[-2:]
    
    fig, axes = plt.subplots(1, len(latest_vals), figsize=(12, 4))
    if len(latest_vals) == 1:
        axes = [axes]
    
    for i, val_file in enumerate(latest_vals):
        try:
            img = Image.open(val_file)
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f"Epoch {os.path.basename(val_file).split('_')[2].split('.')[0]}")
            axes[i].axis('off')
        except Exception as e:
            axes[i].set_title(f"Error: {e}")
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Live monitoring functions loaded!")
print("📊 These functions will show progress BETWEEN training sessions")
print("🔄 During training, progress is shown automatically in the training output")
print("📈 TensorBoard provides real-time metrics during training")
print("\n🔍 Current status:")
monitor_training_progress()


In [ ]:
# Quick checkpoint check and training options
checkpoint_dir = f'{colab_project_path}/checkpoints'
existing_checkpoints = glob.glob(f"{checkpoint_dir}/*.pt") if os.path.exists(checkpoint_dir) else []

print("🔍 Training Status Check:")
if existing_checkpoints:
    latest_checkpoint = max(existing_checkpoints, key=os.path.getctime)
    print(f"✅ Found {len(existing_checkpoints)} checkpoint(s)")
    print(f"📅 Latest: {os.path.basename(latest_checkpoint)}")
    print("\n🎯 Options: Fresh Training | Resume Training")
else:
    print("🆕 No checkpoints found - ready for fresh training")

print(f"\n📊 Config: Batch 32 | Mixed Precision | Max 1000 epochs")
print(f"💾 Save location: {checkpoint_dir}")


In [ ]:
## 4. Training Execution

🔧 Verifying A100 configuration from local path...
✅ Configuration loaded successfully!
📊 Training batch size: 32
⚡ Mixed precision: 16-mixed
🔄 Max epochs: 1000
🎯 Use conditioning: True
💾 Colab optimizations enabled: True

💾 Artifacts (checkpoints, logs, outputs) will be saved to Google Drive at: /content/drive/MyDrive/PKL-DiffusionDenoising


In [ ]:
# 🆕 OPTION 1: Start Fresh Training (With Live Monitoring)
print("🚀 Starting FRESH training session...")
print("📊 Dataset: 9,225 training pairs | Batch: 32 | Mixed precision: 16-bit")
print("🔄 Max epochs: 1000 | Expected time: 8-12 hours")
print("💾 Checkpoints saved every epoch to Google Drive")
print("\n📈 LIVE MONITORING FEATURES:")
print("✅ TensorBoard logs updated in real-time")
print("✅ Validation images saved every epoch (WF | Predicted | 2P GT)")
print("✅ Progress printed every epoch with loss and checkpoint info")
print("✅ Sample images generated every epoch")
print("\n🚀 Starting training now...")

# Run training script with live monitoring built-in
!python {colab_project_path}/scripts/train_real_data.py --config-name=config_colab

🔍 Checking for existing training checkpoints in Google Drive...
🆕 No existing checkpoints found in Google Drive - will start fresh training
✅ Ready to begin new training session!

📊 Training Configuration:
🎯 Batch size: 32 (optimized for A100's 40GB memory)
⚡ Mixed precision: Enabled (16-bit)
🔄 Max epochs: 1000 (with early stopping patience: 5)
⏰ Expected training time: 8-12 hours (full training to convergence)
💾 Checkpoints saved every epoch to Google Drive at: /content/drive/MyDrive/PKL-DiffusionDenoising/checkpoints
📈 Validation on 1,125 samples every epoch


In [ ]:
# 🔄 OPTION 2: Resume Training (With Live Monitoring)
import torch

print("🔄 Resuming training from latest checkpoint...")

checkpoints = glob.glob(f"{checkpoint_dir}/*.pt")
if not checkpoints:
    print("❌ No checkpoints found! Please run 'Start Fresh Training' instead.")
else:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"📅 Resuming from: {os.path.basename(latest_checkpoint)}")
    
    # Quick checkpoint info
    try:
        ckpt_data = torch.load(latest_checkpoint, map_location='cpu', weights_only=True)
        epoch = ckpt_data.get('epoch', '?')
        val_loss = ckpt_data.get('last_val_loss', '?')
        print(f"📈 Last epoch: {epoch} | Loss: {val_loss}")
        print(f"⏰ Remaining: {1000 - epoch if isinstance(epoch, int) else '?'} epochs")
    except Exception as e:
        print(f"⚠️  Could not read checkpoint metadata: {e}")
    
    print("\n🚀 Continuing training with live monitoring...")
    print("📊 Batch: 32 | Mixed precision: 16-bit | Max epochs: 1000")
    print("\n📈 LIVE MONITORING FEATURES:")
    print("✅ TensorBoard logs updated in real-time")
    print("✅ Validation images saved every epoch (WF | Predicted | 2P GT)")
    print("✅ Progress printed every epoch with loss and checkpoint info")
    print("\n🚀 Resuming training now...")
    
    # Run training script with resume
    !python {colab_project_path}/scripts/train_real_data.py --config-name=config_colab


## 5. Monitoring Between Training Sessions


In [ ]:
# Run this cell BETWEEN training sessions to check progress
print("🔄 Checking training progress...")
print("ℹ️  Note: During training, progress is shown automatically in the training output")
monitor_training_progress()


In [ ]:
# Run this cell BETWEEN training sessions to view latest validation images
print("🖼️ Displaying latest validation visualizations...")
print("ℹ️  Note: Validation images are automatically saved every epoch during training")
display_latest_validation()


## 6. Download Results


In [ ]:
# Download results
from google.colab import files
import zipfile

zip_path = '/content/training_results.zip'
print(f"📦 Creating results zip from: {colab_project_path}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for dir_name in ['checkpoints', 'outputs', 'logs']:
        dir_path = f'{colab_project_path}/{dir_name}'
        if os.path.exists(dir_path):
            print(f"Adding {dir_name}...")
            for root, dirs, files in os.walk(dir_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, colab_project_path)
                    zipf.write(file_path, arc_path)

print(f"📁 Zip size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
files.download(zip_path)
print("✅ Results downloaded!")


In [ ]:
## 5. Training Execution

🚀 Starting FRESH training session...
⚠️  This will ignore any existing checkpoints and start from epoch 0
📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)
🎯 Batch size: 32 (optimized for A100's 40GB memory)
⚡ Mixed precision: Enabled (16-bit)
🔄 Max epochs: 1000 (with early stopping patience: 5)
⏰ Expected training time: 8-12 hours (full training to convergence)
💾 Checkpoints will be saved every epoch to Google Drive
📈 Validation on 1,125 samples every epoch

🚀 Starting training now...
2025-09-11 03:36:27.085279: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-11 03:36:27.104085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been re

# Check for existing checkpoints and training options
checkpoint_dir = f'{colab_project_path}/checkpoints'
existing_checkpoints = glob.glob(f"{checkpoint_dir}/*.pt") if os.path.exists(checkpoint_dir) else []

print("🔍 Checking for existing training checkpoints...")
if existing_checkpoints:
    latest_checkpoint = max(existing_checkpoints, key=os.path.getctime)
    print(f"✅ Found {len(existing_checkpoints)} existing checkpoint(s)")
    print(f"📅 Latest checkpoint: {os.path.basename(latest_checkpoint)}")
    print("\n🎯 Training Options:")
    print("1. 🆕 Start Fresh Training (ignore existing checkpoints)")
    print("2. 🔄 Resume Training (continue from latest checkpoint)")
else:
    print("🆕 No existing checkpoints found - will start fresh training")
    print("✅ Ready to begin new training session!")

print("\n📊 Training Configuration:")
print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
print("⚡ Mixed precision: Enabled (16-bit)")
print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
print("⏰ Expected training time: 8-12 hours (full training to convergence)")
print(f"💾 Checkpoints saved every epoch to Google Drive at: {checkpoint_dir}")
print("📈 Validation on 1,125 samples every epoch")

# 🆕 OPTION 1: Start Fresh Training (Ignore Existing Checkpoints)
import subprocess
import sys

print("🚀 Starting FRESH training session...")
print("⚠️  This will ignore any existing checkpoints and start from epoch 0")
print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
print("⚡ Mixed precision: Enabled (16-bit)")
print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
print("⏰ Expected training time: 8-12 hours (full training to convergence)")
print("💾 Checkpoints will be saved every epoch to Google Drive")
print("📈 Validation on 1,125 samples every epoch")
print("\n🚀 Starting training now...")

# Run training script
!python {colab_project_path}/scripts/train_real_data.py --config-name=config_colab

# 🔄 OPTION 2: Resume Training (Continue from Latest Checkpoint)
import torch

print("🔄 Resuming training from latest checkpoint...")

# Find the latest checkpoint
checkpoints = glob.glob(f"{checkpoint_dir}/*.pt")

if not checkpoints:
    print("❌ No checkpoints found! Please run 'Start Fresh Training' instead.")
else:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"📅 Resuming from: {os.path.basename(latest_checkpoint)}")
    
    # Try to load checkpoint info
    try:
        ckpt_data = torch.load(latest_checkpoint, map_location='cpu')
        epoch = ckpt_data.get('epoch', 'unknown')
        val_loss = ckpt_data.get('last_val_loss', 'unknown')
        global_step = ckpt_data.get('global_step', 'unknown')
        
        print(f"📈 Last epoch: {epoch}")
        print(f"📊 Last validation loss: {val_loss}")
        print(f"🔢 Global step: {global_step}")
        print(f"⏰ Remaining epochs: {1000 - epoch if isinstance(epoch, int) else 'unknown'}")
    except Exception as e:
        print(f"⚠️  Could not read checkpoint metadata: {e}")
    
    print("\n🚀 Continuing training with A100 optimizations...")
    print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
    print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
    print("⚡ Mixed precision: Enabled (16-bit)")
    print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
    print("💾 Checkpoints will be saved every epoch to Google Drive")
    print("📈 Validation on 1,125 samples every epoch")
    print("\n🚀 Resuming training now...")
    
    # Run training script with resume
    !python {colab_project_path}/scripts/train_real_data.py --config-name=config_colab

## 6. Live Monitoring During Training


In [ ]:
# Run this cell periodically during training to monitor progress
print("🔄 Checking training progress...")
monitor_training_progress()


In [ ]:
# Run this cell to display latest validation images
print("🖼️ Displaying latest validation visualizations...")
display_latest_validation()


## 7. Download Results


In [ ]:
# Download results to local machine
from google.colab import files
import zipfile
import shutil

def create_results_zip():
    """Create a zip file with all training results."""
    zip_path = '/content/training_results.zip'
    
    print(f"📦 Creating results zip from: {colab_project_path}")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add checkpoints
        checkpoint_dir = f'{colab_project_path}/checkpoints'
        if os.path.exists(checkpoint_dir):
            print(f"Adding checkpoints from {checkpoint_dir}")
            for root, dirs, files in os.walk(checkpoint_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, colab_project_path)
                    zipf.write(file_path, arc_path)
        
        # Add outputs
        outputs_dir = f'{colab_project_path}/outputs'
        if os.path.exists(outputs_dir):
            print(f"Adding outputs from {outputs_dir}")
            for root, dirs, files in os.walk(outputs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, colab_project_path)
                    zipf.write(file_path, arc_path)
        
        # Add logs
        logs_dir = f'{colab_project_path}/logs'
        if os.path.exists(logs_dir):
            print(f"Adding logs from {logs_dir}")
            for root, dirs, files in os.walk(logs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arc_path = os.path.relpath(file_path, colab_project_path)
                    zipf.write(file_path, arc_path)
    
    return zip_path

# Create and download results
try:
    zip_path = create_results_zip()
    print(f"📦 Results zip created: {zip_path}")
    print(f"📁 Zip size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
    
    # Download the zip file
    files.download(zip_path)
    print("✅ Results downloaded to your local machine!")
    print("📊 Includes: checkpoints, validation visualizations, TensorBoard logs, and samples")
    
except Exception as e:
    print(f"❌ Failed to create or download results zip: {e}")
    print("Ensure Google Drive is mounted and the artifact path exists.")


In [ ]:
# 🔄 OPTION 2: Resume Training (Continue from Latest Checkpoint)
import torch

print("🔄 Resuming training from latest checkpoint...")

# Set the PROJECT_ROOT environment variable to the local Colab path
colab_project_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
os.environ['PROJECT_ROOT'] = colab_project_path

# Find the latest checkpoint in the Google Drive artifact directory
checkpoint_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/checkpoints'
checkpoints = glob.glob(f"{checkpoint_dir}/*.pt")

if not checkpoints:
    print("❌ No checkpoints found! Please run 'Start Fresh Training' instead.")
else:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"📅 Resuming from: {os.path.basename(latest_checkpoint)}")

    # Try to load checkpoint info
    try:
        ckpt_data = torch.load(latest_checkpoint, map_location='cpu')
        epoch = ckpt_data.get('epoch', 'unknown')
        val_loss = ckpt_data.get('last_val_loss', 'unknown')
        global_step = ckpt_data.get('global_step', 'unknown')

        print(f"📈 Last epoch: {epoch}")
        print(f"📊 Last validation loss: {val_loss}")
        print(f"🔢 Global step: {global_step}")
        print(f"⏰ Remaining epochs: {1000 - epoch if isinstance(epoch, int) else 'unknown'}")

    except Exception as e:
        print(f"⚠️  Could not read checkpoint metadata: {e}")

    print("\n🚀 Continuing training with A100 optimizations...")
    print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
    print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
    print("⚡ Mixed precision: Enabled (16-bit)")
    print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
    print("💾 Checkpoints will be saved every epoch to Google Drive")
    print("📈 Validation on 1,125 samples every epoch")
    print("\n🚀 Resuming training now...")

    # Run training script with resume from the local Colab path
    # The script should be configured (via OmegaConf) to load the checkpoint
    # and save artifacts to the Drive path
    !python {colab_project_path}/scripts/train_real_data.py --config-name=config_colab

## 3. Training Execution

Choose your training mode:
- **🆕 Start Fresh Training**: Begin a new training session from scratch (ignores existing checkpoints)
- **🔄 Resume Training**: Continue training from the latest checkpoint


In [ ]:
# Start TensorBoard for live monitoring
import subprocess
import threading
import time
import os

def start_tensorboard():
    """Start TensorBoard in the background."""
    # TensorBoard logs are stored in Google Drive for persistence
    logs_dir = '/content/drive/MyDrive/PKL-DiffusionDenoising/logs'
    os.makedirs(logs_dir, exist_ok=True)

    # Start TensorBoard
    cmd = ['tensorboard', '--logdir', logs_dir, '--port', '6006', '--host', '0.0.0.0']
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # Wait a moment for TensorBoard to start
    time.sleep(3)

    if process.poll() is None:
        print("✅ TensorBoard started successfully!")
        print("📊 View live training metrics at:")
        print("   https://colab.research.google.com/github/tensorflow/tensorboard/blob/master/tensorboard/notebooks/tensorboard_in_notebooks.ipynb")
        print("   Or use: !tensorboard --logdir /content/drive/MyDrive/PKL-DiffusionDenoising/logs --port 6006")
        return process
    else:
        print("❌ Failed to start TensorBoard")
        return None

# Start TensorBoard
tb_process = start_tensorboard()

In [ ]:
# Alternative: Embed TensorBoard directly in notebook
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/PKL-DiffusionDenoising/logs --port 6006

## 4. Live Training Monitoring


In [ ]:
# Setup Weights & Biases (optional)
import wandb

# Login to W&B (you'll need to get your API key from https://wandb.ai/settings)
# wandb.login()

print("📊 Weights & Biases setup complete")


In [ ]:
# Combined Monitoring Section
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch
import os

def monitor_combined_training():
    """Monitor training progress and display latest results."""
    # Define artifact paths in Google Drive
    drive_artifact_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
    checkpoint_dir = f'{drive_artifact_path}/checkpoints'
    logs_dir = f'{drive_artifact_path}/logs'
    samples_dir = f'{drive_artifact_path}/outputs/samples' # Assuming samples are saved here

    print("📊 Combined Training Progress Monitor")
    print("=" * 60)
    print("🎯 Batch size: 32 | ⚡ Mixed precision: Enabled")
    print("📊 Dataset: 9,225 training pairs | 1,125 validation pairs")
    print("🔄 Max epochs: 1000 | Typical diffusion training: 800-1000 epochs")
    print("⏰ Expected time: 8-12 hours (full training to convergence)")
    # The previous run info is from a previous execution, keep it for context
    print("📈 Previous run: 300 epochs completed, continuing to full convergence")
    print("=" * 60)

    # Check for checkpoints in Google Drive
    checkpoints = glob.glob(f"{checkpoint_dir}/epoch_*.pt")
    if checkpoints:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f"💾 Latest checkpoint: {os.path.basename(latest_ckpt)}")
        try:
            ckpt_data = torch.load(latest_ckpt, map_location='cpu')
            print(f"📈 Epoch: {ckpt_data.get('epoch', 'unknown')}")
            print(f"📊 Last validation loss: {ckpt_data.get('last_val_loss', 'unknown')}")
            print(f"🔢 Global step: {ckpt_data.get('global_step', 'unknown')}")
        except Exception as e:
            print(f"⚠️  Could not read checkpoint metadata from {latest_ckpt}: {e}")
    else:
        print("⏳ No checkpoints found yet in Google Drive...")

    # Check for sample images in Google Drive outputs
    if os.path.exists(samples_dir):
        sample_files = glob.glob(f"{samples_dir}/*/epoch_*.png")
        if sample_files:
            latest_sample = max(sample_files, key=os.path.getctime)
            print(f"🎨 Latest sample: {os.path.basename(latest_sample)}")
        else:
            print("🖼️ No sample images found in Google Drive outputs...")

        # Check for validation visualizations
        val_files = glob.glob(f"{samples_dir}/*/validation_epoch_*.png")
        if val_files:
            latest_val = max(val_files, key=os.path.getctime)
            print(f"📊 Latest validation: {os.path.basename(latest_val)}")
        else:
            print("🖼️ No validation visualizations found in Google Drive outputs...")
    else:
         print(f"📁 Outputs directory not found yet in Google Drive: {samples_dir}")

    # Check TensorBoard logs in Google Drive
    if os.path.exists(logs_dir):
        log_files = glob.glob(f"{logs_dir}/*/events.out.tfevents.*")
        if log_files:
            print(f"📈 TensorBoard logs found in Google Drive: {len(log_files)} files")
        else:
             print("📉 No TensorBoard log files found in Google Drive...")
    else:
         print(f"📁 Logs directory not found yet in Google Drive: {logs_dir}")

    print("=" * 60)

# Run combined monitoring
monitor_combined_training()

In [ ]:
# Display A100-optimized training configuration
import yaml
from omegaconf import OmegaConf

# Load Colab config and training config separately
cfg = OmegaConf.load('configs/config_colab.yaml')
training_cfg = OmegaConf.load('configs/training/ddpm_colab.yaml')

print("🔧 A100 Training Configuration:")
print(OmegaConf.to_yaml(training_cfg))

print("\n🚀 A100 Optimizations:")
if hasattr(training_cfg, 'colab_optimizations'):
    print(OmegaConf.to_yaml(training_cfg.colab_optimizations))

print("\n📊 Dataset Information:")
print(f"Training samples: 9,225 (41 frames × 225 patches/frame)")
print(f"Validation samples: 1,125 (5 frames × 225 patches/frame)")
print(f"Test samples: 1,350 (6 frames × 225 patches/frame)")
print(f"Image size: 256×256 pixels")

print("\n⏰ Performance Estimates:")
print(f"Batch size: {training_cfg.batch_size}")
print(f"Iterations per epoch: {9225 // training_cfg.batch_size}")
print(f"Max epochs: {training_cfg.max_epochs}")
print(f"Early stopping patience: {training_cfg.early_stopping_patience} epochs")
print(f"Expected training time: 8-12 hours (typical diffusion training: 800-1000 epochs)")
print(f"Total possible iterations: {9225 // training_cfg.batch_size * training_cfg.max_epochs}")
print(f"Previous run: 300 epochs completed, continuing to full convergence")

print("\n📊 Logging Configuration:")
print(f"TensorBoard logs: {cfg.paths.logs}")
print(f"Checkpoints: {cfg.paths.checkpoints}")
print(f"Outputs: {cfg.paths.outputs}")
print(f"W&B project: {cfg.wandb.project}")


In [ ]:
# Start training with A100 optimizations
import subprocess
import sys

# Set environment variables for Colab
os.environ['PROJECT_ROOT'] = '/content/PKL-DiffusionDenoising'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Display training configuration
print("🚀 Starting training with A100 optimizations...")
print("📊 Dataset: 9,225 training pairs (41 frames × 225 patches/frame)")
print("🎯 Batch size: 32 (optimized for A100's 40GB memory)")
print("⚡ Mixed precision: Enabled (16-bit)")
print("🔄 Max epochs: 1000 (with early stopping patience: 5)")
print("⏰ Expected training time: 4-6 hours (early stopping around epoch 50-100)")
print("💾 Checkpoints saved every epoch to Google Drive")
print("📈 Validation on 1,125 samples every epoch")
print("🔢 Total possible iterations: 289,000 (289 per epoch × 1000 epochs)")

# Run training script
result = subprocess.run([
    sys.executable, 'scripts/train_real_data.py',
    '--config-name=config_colab'
], capture_output=True, text=True)

print("Training completed!")
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


## 6. Real-time Monitoring


In [ ]:
# Monitor A100 training progress in real-time
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch # Import torch for loading checkpoints

def monitor_a100_training():
    """Monitor A100 training progress and display latest results."""
    # Define artifact paths in Google Drive
    drive_artifact_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
    checkpoint_dir = f'{drive_artifact_path}/checkpoints'
    logs_dir = f'{drive_artifact_path}/logs'
    samples_dir = f'{drive_artifact_path}/outputs/samples' # Assuming samples are saved here

    print("📊 A100 Training Progress Monitor")
    print("=" * 60)
    # Note: The configuration details below are from the original repo and may not apply.
    print("🎯 Batch size: 32 | ⚡ Mixed precision: Enabled")
    print("📊 Dataset: 9,225 training pairs | 1,125 validation pairs")
    print("🔄 Max epochs: 1000 | Typical diffusion training: 800-1000 epochs")
    print("⏰ Expected time: 8-12 hours (full training to convergence)")
    print("📈 Previous run: 300 epochs completed, continuing to full convergence")
    print("=" * 60)

    # Check for checkpoints in Google Drive
    # Note: Checkpoint naming convention might differ in openai/guided-diffusion.
    checkpoints = glob.glob(f"{checkpoint_dir}/epoch_*.pt") # Assuming epoch_*.pt naming
    if checkpoints:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f"💾 Latest checkpoint: {os.path.basename(latest_ckpt)}")

        # Load checkpoint info
        try:
            # Note: Checkpoint structure might differ. Loading might fail or yield different keys.
            ckpt_data = torch.load(latest_ckpt, map_location='cpu')
            print(f"📈 Epoch: {ckpt_data.get('epoch', 'unknown')}")
            print(f"📊 Last validation loss: {ckpt_data.get('last_val_loss', 'unknown')}") # Key might be different
            print(f"🔢 Global step: {ckpt_data.get('global_step', 'unknown')}") # Key might be different
        except Exception as e:
            print(f"⚠️  Could not read checkpoint metadata from {latest_ckpt}: {e}")


    else:
        print("⏳ No checkpoints found yet in Google Drive...")

    # Check for sample images in Google Drive outputs
    if os.path.exists(samples_dir):
        # Note: Sample naming and location might differ in openai/guided-diffusion.
        sample_files = glob.glob(f"{samples_dir}/*/epoch_*.png") # Assuming directory structure and naming
        if sample_files:
            latest_sample = max(sample_files, key=os.path.getctime)
            print(f"🎨 Latest sample: {os.path.basename(latest_sample)}")

        # Check for validation visualizations
        # Note: Validation visualization naming and location might differ.
        val_files = glob.glob(f"{samples_dir}/*/validation_epoch_*.png") # Assuming naming
        if val_files:
            latest_val = max(val_files, key=os.path.getctime)
            print(f"📊 Latest validation: {os.path.basename(latest_val)}")
        else:
            print("🖼️ No validation visualizations found in Google Drive outputs...")
    else:
         print(f"📁 Outputs directory not found yet in Google Drive: {samples_dir}")

    # Check TensorBoard logs in Google Drive
    if os.path.exists(logs_dir):
        # Note: Log file naming might differ.
        log_files = glob.glob(f"{logs_dir}/*/events.out.tfevents.*") # Standard TensorBoard naming
        if log_files:
            print(f"📈 TensorBoard logs found in Google Drive: {len(log_files)} files")
        else:
             print("📉 No TensorBoard log files found in Google Drive...")
    else:
         print(f"📁 Logs directory not found yet in Google Drive: {logs_dir}")


    print("=" * 60)

# Run monitoring
monitor_a100_training()

In [ ]:
# Display latest validation visualizations (WF | Predicted | 2P GT)
import time
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import os # Import os

def display_latest_validation():
    """Display the latest validation visualization from Google Drive."""
    drive_artifact_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
    samples_dir = f'{drive_artifact_path}/outputs/samples' # Assuming samples are saved here

    # Find all sample directories in Google Drive outputs
    sample_dirs = glob.glob(f"{samples_dir}/*")
    if not sample_dirs:
        print("❌ No sample directories found in Google Drive outputs")
        return

    # Get the latest experiment directory (assuming structure similar to original)
    latest_dir = max(sample_dirs, key=os.path.getctime)
    print(f"🔍 Checking latest experiment directory in Google Drive: {latest_dir}")

    # Find validation visualization files
    # Note: Validation visualization naming and location might differ.
    val_files = glob.glob(f"{latest_dir}/validation_epoch_*.png") # Assuming naming
    if not val_files:
        print(f"❌ No validation visualizations found in {latest_dir}")
        return

    # Display the latest few validation visualizations
    # Sort by epoch number extracted from filename
    try:
        val_files.sort(key=lambda x: int(os.path.basename(x).split('_')[2].split('.')[0]))
        latest_vals = val_files[-3:]  # Show last 3 validation epochs
    except Exception as e:
        print(f"⚠️  Could not sort validation files by epoch: {e}. Displaying last 3 files by creation time.")
        latest_vals = sorted(val_files, key=os.path.getctime)[-3:]


    fig, axes = plt.subplots(1, len(latest_vals), figsize=(18, 6))
    if len(latest_vals) == 1:
        axes = [axes]

    for i, val_file in enumerate(latest_vals):
        try:
            img = Image.open(val_file)
            axes[i].imshow(img, cmap='gray')
            # Extract epoch number (might fail if naming is different)
            try:
                epoch_num = os.path.basename(val_file).split('_')[2].split('.')[0]
                axes[i].set_title(f"Epoch {epoch_num}\n(WF | Predicted | 2P GT - assumed)")
            except:
                 axes[i].set_title(f"Validation: {os.path.basename(val_file)}\n(WF | Predicted | 2P GT - assumed)")
            axes[i].axis('off')
        except Exception as e:
            print(f"❌ Could not display image {val_file}: {e}")
            axes[i].set_title(f"Error loading {os.path.basename(val_file)}")
            axes[i].axis('off')


    plt.tight_layout()
    plt.show()

    print(f"📊 Showing validation visualizations from Google Drive: {latest_dir}")

# Display validation visualizations
# Note: This will only work if validation images are being saved in the expected location and format
display_latest_validation()

## 7. Resume Training


In [ ]:
# Resume training from latest checkpoint
import subprocess
import sys
import os

print("🔄 Resuming training from latest checkpoint...")
print("📊 Will continue with A100 optimizations (batch size 32)")
print("⏰ Remaining training time depends on convergence")

# Set environment variables for Colab
colab_project_path = '/content/PKL-DiffusionDenoising'
os.environ['PROJECT_ROOT'] = colab_project_path
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # Assuming using GPU 0

# Define Google Drive artifact path for checkpoints
drive_artifact_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
checkpoint_dir = f'{drive_artifact_path}/checkpoints'

# Note: The training script path and arguments are based on the original repo.
# The script should be configured (via OmegaConf) to load the checkpoint
# and save artifacts to the Drive path.
!python {colab_project_path}/scripts/train_diffusion.py --config-name=config_colab

## 8. Download Results


In [ ]:
# Download results to local machine
from google.colab import files
import zipfile
import shutil
import os # Import os module

def create_results_zip():
    """Create a zip file with all training results from Google Drive artifacts."""
    drive_artifact_path = '/content/drive/MyDrive/PKL-DiffusionDenoising'
    zip_path = '/content/training_results.zip' # Zip file created locally in Colab

    print(f"📦 Creating results zip from Google Drive artifacts: {drive_artifact_path}")

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add checkpoints
        checkpoint_dir = f'{drive_artifact_path}/checkpoints'
        if os.path.exists(checkpoint_dir):
            print(f"Adding checkpoints from {checkpoint_dir}")
            for root, dirs, files in os.walk(checkpoint_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Store in zip relative to the artifact path
                    arc_path = os.path.relpath(file_path, drive_artifact_path)
                    zipf.write(file_path, arc_path)
        else:
             print(f"Checkpoints directory not found: {checkpoint_dir}")

        # Add outputs (including validation visualizations)
        outputs_dir = f'{drive_artifact_path}/outputs'
        if os.path.exists(outputs_dir):
            print(f"Adding outputs from {outputs_dir}")
            for root, dirs, files in os.walk(outputs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Store in zip relative to the artifact path
                    arc_path = os.path.relpath(file_path, drive_artifact_path)
                    zipf.write(file_path, arc_path)
        else:
             print(f"Outputs directory not found: {outputs_dir}")

        # Add logs
        logs_dir = f'{drive_artifact_path}/logs'
        if os.path.exists(logs_dir):
             print(f"Adding logs from {logs_dir}")
             for root, dirs, files in os.walk(logs_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Store in zip relative to the artifact path
                    arc_path = os.path.relpath(file_path, drive_artifact_path)
                    zipf.write(file_path, arc_path)
        else:
             print(f"Logs directory not found: {logs_dir}")


    return zip_path

# Create and download results
# Note: This will attempt to zip files from Google Drive.
# If Drive is not mounted or paths are incorrect, it might fail or zip empty directories.
try:
    zip_path = create_results_zip()
    print(f"📦 Results zip created: {zip_path}")
    print(f"📁 Zip size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")

    # Download the zip file
    files.download(zip_path)
    print("✅ Results downloaded to your local machine!")
    print("📊 Includes: checkpoints, validation visualizations, TensorBoard logs, and samples (from Google Drive)")

except Exception as e:
    print(f"❌ Failed to create or download results zip: {e}")
    print("Ensure Google Drive is mounted and the artifact path exists and is accessible.")